# 04 — Extract q-projection and generate a separate accumulator fixture
Read the matching frozen TFLite model. Use the original synthetic seed-42 block input,
extract the intermediate activation, and compute the mathematical INT32 accumulator.
Save a new NPZ under `artifacts/exports/reexports/<run-id>/`, generated C under
`src/c/generated/<run-id>/`, and quantization metadata/manifest under `results/notebook_runs/<run-id>/`.
The generated C test checks accumulators only. `y_ref` is exported for separate requantization checks.
The measured INT8/Q39 baseline remains in `experiments/block0_qproj/` and is preserved.

Start from a fresh kernel and run top to bottom. The final cell records output hashes; incomplete runs remain `started`.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter in the extracted tinystories-rv64imc folder.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import (MODEL_ID, MODEL_REVISION, SEQUENCE_LENGTH,
                             MODEL_DIR, EXPORT_DIR, RESULT_DIR, C_DIR, MODEL_PATH,
                             PROMPT, GENERATED_LENGTH)
from notebook_runs import start_run, finish_run
RUN_DIR = start_run(ROOT, 'notebooks/04_export_qproj_to_c.ipynb', {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'sequence_length': SEQUENCE_LENGTH}, inputs=[MODEL_PATH])
RESULT_DIR = RUN_DIR
EXPORT_DIR = ROOT / 'artifacts/exports/reexports' / RUN_DIR.name
C_DIR = ROOT / 'src/c/generated' / RUN_DIR.name


In [ ]:
import hashlib
import json
import numpy as np
from litert_helpers import reference_interpreter
reference = reference_interpreter(MODEL_PATH)

## Extract the original operation
This extraction is deliberately specific to the verified no-bias q-projection baseline.
If an assertion fails after reconversion, inspect notebook 03 and confirm the operation's
identity before changing indices. Matching dimensions alone cannot distinguish Q, K, and V.

In [ ]:
# Confirm we are extracting the operation identified in your output.
op = next(
    op for op in reference._get_ops_details()
    if op["index"] == 15
)
assert op["op_name"] == "FULLY_CONNECTED"
assert list(op["inputs"]) == [45, 27, -1]
assert list(op["outputs"]) == [46]

inputs = reference.get_input_details()
assert len(inputs) == 1
assert inputs[0]["shape"].tolist() == [1, 32, 128]
assert inputs[0]["dtype"] == np.float32

# Deterministic synthetic block input for the first arithmetic test.
# Later, repeat with real hidden states from your token-based test.
rng = np.random.default_rng(42)
block_input = rng.standard_normal((1, 32, 128)).astype(np.float32)

reference.set_tensor(inputs[0]["index"], block_input)
reference.invoke()

# Capture the projection's input AFTER the preceding operations.
x_q = reference.get_tensor(45).copy()
w_q = reference.get_tensor(27).copy()
y_ref = reference.get_tensor(46).copy()

assert x_q.dtype == w_q.dtype == y_ref.dtype == np.int8
assert x_q.shape == y_ref.shape == (1, 32, 128)
assert w_q.shape == (128, 128)

details = {
    t["index"]: t
    for t in reference.get_tensor_details()
}

qx = details[45]["quantization_parameters"]
qw = details[27]["quantization_parameters"]
qy = details[46]["quantization_parameters"]

assert qx['scales'].size == qx['zero_points'].size == 1
assert qy['scales'].size == qy['zero_points'].size == 1
assert qw['scales'].size in (1, 128)
assert qw['scales'].size == 1 or int(qw['quantized_dimension']) == 0
zx = int(qx["zero_points"][0])
assert np.all(qw["zero_points"] == 0)

# Mathematical INT32 accumulator reference, before requantization.
# Cast BEFORE subtracting the zero point to avoid INT8 overflow.
x_centered = x_q.reshape(32, 128).astype(np.int32) - zx
acc64 = x_centered.astype(np.int64) @ w_q.astype(np.int64).T
assert np.all(acc64 >= np.iinfo(np.int32).min) and np.all(acc64 <= np.iinfo(np.int32).max)
acc_ref = acc64.astype(np.int32)

export_dir = EXPORT_DIR
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / "q_proj_case0.npz"

np.savez(
    export_path,
    block_input=block_input,
    x_q=x_q,
    w_q=w_q,
    acc_ref=acc_ref,
    y_ref=y_ref,
    x_scales=qx["scales"],
    x_zero_points=qx["zero_points"],
    w_scales=qw["scales"],
    w_zero_points=qw["zero_points"],
    w_quantized_dimension=np.array(qw["quantized_dimension"]),
    y_scales=qy["scales"],
    y_zero_points=qy["zero_points"],
)

print("Saved:", export_path)
print("x_q:", x_q.shape, x_q.dtype)
print("w_q:", w_q.shape, w_q.dtype)
print("acc_ref:", acc_ref.shape, acc_ref.dtype)
print("y_ref:", y_ref.shape, y_ref.dtype)
print("First 8 accumulators:", acc_ref[0, :8])
print("First 8 reference outputs:", y_ref[0, 0, :8])

## Calculate and save the actual requantization multipliers

In [ ]:
with np.load(EXPORT_DIR / 'q_proj_case0.npz') as data:
    sx = float(data['x_scales'][0])
    sw = data['w_scales'].astype(np.float64)
    sy = float(data['y_scales'][0])
    zy = int(data['y_zero_points'][0])
    assert sx > 0 and sy > 0 and np.all(sw > 0)
    multipliers = sx * sw / sy
    report = {
        'model_sha256': hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
        'seed': 42, 'input_kind': 'synthetic_normal_block_input',
        'op_index': 15, 'x_tensor': 45, 'weight_tensor': 27, 'y_tensor': 46,
        'x_scale': sx, 'x_zero_point': zx, 'weight_scales': sw.tolist(),
        'y_scale': sy, 'y_zero_point': zy,
        'multipliers': multipliers.tolist(),
        'all_multipliers_equal': bool(np.all(multipliers == multipliers[0])),
    }
RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / 'qproj_case0.json').write_text(json.dumps(report, indent=2))
print('Weight-scale entries:', sw.size)
print('All multipliers equal:', report['all_multipliers_equal'])
print('First four multipliers:', multipliers[:4])
print('Output zero point:', zy)

## Generate the accumulator C test and both reference headers

In [ ]:
with np.load(EXPORT_DIR / "q_proj_case0.npz") as data:
    x = data["x_q"].reshape(32, 128)
    w = data["w_q"]
    expected = data["acc_ref"]
    zx = int(data["x_zero_points"][0])

    expected_y = data["y_ref"].reshape(-1)

    assert x.shape == (32, 128)
    assert w.shape == (128, 128)
    assert expected.shape == (32, 128)
    assert np.all(data["w_zero_points"] == 0)

def c_array(name, dtype, values):
    flat = values.reshape(-1)
    lines = [
        "    " + ", ".join(str(int(v)) for v in flat[i:i+16])
        for i in range(0, flat.size, 16)
    ]
    return (
        f"static const {dtype} {name}[{flat.size}] = {{\n"
        + ",\n".join(lines) + "\n};\n"
    )

header = (
    "#ifndef QPROJ_DATA_H\n"
    "#define QPROJ_DATA_H\n"
    "#include <stdint.h>\n"
    f"#define INPUT_ZERO_POINT ({zx})\n\n"
)
header += c_array("input", "int8_t", x)
header += c_array("weights", "int8_t", w)
header += c_array("expected_acc", "int32_t", expected)
header += "#endif\n"

C_DIR.mkdir(parents=True, exist_ok=True)
(C_DIR / "qproj_data.h").write_text(header)
y_header = '#ifndef QPROJ_Y_REF_H\n#define QPROJ_Y_REF_H\n#include <stdint.h>\n'
y_header += c_array('expected_y', 'int8_t', expected_y)
y_header += '#endif\n'
(C_DIR / 'qproj_y_ref.h').write_text(y_header)

source = r'''
#include <stdint.h>
#include <stdio.h>
#include "qproj_data.h"

static int32_t output[32 * 128];

__attribute__((noinline))
void qproj_accumulate(void)
{
    for (int t = 0; t < 32; ++t) {
        for (int o = 0; o < 128; ++o) {
            int32_t acc = 0;

            for (int k = 0; k < 128; ++k) {
                int32_t x =
                    (int32_t)input[t * 128 + k]
                    - INPUT_ZERO_POINT;

                int32_t w =
                    (int32_t)weights[o * 128 + k];

                acc += x * w;
            }

            output[t * 128 + o] = acc;
        }
    }
}

int main(void)
{
    qproj_accumulate();

    int errors = 0;
    for (int i = 0; i < 32 * 128; ++i) {
        if (output[i] != expected_acc[i])
            ++errors;
    }

    printf("Checked 4096 accumulators; mismatches: %d\n", errors);
    printf("%s\n", errors == 0 ? "PASS" : "FAIL");

    return errors == 0 ? 0 : 1;
}
'''

(C_DIR / "qproj_acc_test.c").write_text(source)
print("Created qproj_data.h and qproj_acc_test.c")

## Compile and run on the host
Run from the repository root:
```bash
gcc -O2 -Wall -Wextra -I src/c/generated/<run-id> src/c/generated/<run-id>/qproj_acc_test.c -o /tmp/qproj_acc_check
/tmp/qproj_acc_check
```
The executable must report `Checked 4096 accumulators; mismatches: 0` and `PASS`.
This generated test has no requantization implementation, CSR counters, or custom instructions.
Use your existing measured C kernel for subsequent Spike profiling and ISA work.
The assignment distinguishes Spike instruction counts from CV-Wally hardware cycle measurements.

In [ ]:
finish_run(ROOT, RUN_DIR, [EXPORT_DIR / 'q_proj_case0.npz', RESULT_DIR / 'qproj_case0.json', C_DIR / 'qproj_acc_test.c', C_DIR / 'qproj_data.h', C_DIR / 'qproj_y_ref.h'])
print('Saved run record:', RUN_DIR / 'run.json')
